# M2 — Empirical Complexity via Python→Rust Translations (UFC/BJJ edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paiml/big-o-python-to-rust/blob/main/notebooks/m2-empirical.ipynb)

One canonical example per complexity class, each anchored to a course Python→Rust translation lesson. Domain: a `Roster` of UFC fighters. Operation-count proofs (deterministic, Colab-friendly) replace criterion wall-clock for the asserts; the Rust `m2-empirical` crate handles the statistical-CI side via criterion benches. Course lessons 2.1.1 (list comp -> iterator), 2.2.1 (dict -> HashMap), 2.3.1 (sorted -> sort_unstable).

## Lesson 2.2.1 — O(1) — `x in dict` -> `HashMap` lookup

Python dict lookup is amortized O(1); Rust HashMap is the direct translation.
Operation count is one hash + one slot probe — independent of roster size.

In [1]:
roster = {
    "Khabib": 2200,
    "McGregor": 2050,
    "Adesanya": 2150,
    "Jones": 2300,
    "Silva": 2100,
}
assert roster["Khabib"] == 2200
assert roster.get("Unknown") is None

# Operation count is constant: one lookup regardless of roster size
small = {f"fighter_{i}": 2000 + i for i in range(10)}
big = {f"fighter_{i}": 2000 + i for i in range(1_000_000)}
assert small["fighter_5"] == 2005
assert big["fighter_500000"] == 502000
print("constant     : Elo(fighter_500000 of 10^6 roster) = 502000")

constant     : Elo(fighter_500000 of 10^6 roster) = 502000


## O(log n) — `bisect` -> `slice::binary_search`

Binary search on a sorted weight roster locates the boundary between weight
classes in O(log n). The operation count is at most `ceil(log2(n)) + 1`.

In [2]:
from math import ceil, log2


def weight_class_boundary(
    sorted_weights: list[int], target: int
) -> tuple[int | None, int]:
    """Return (boundary_index_or_None, comparison_count)."""
    lo, hi, ops = 0, len(sorted_weights), 0
    while lo < hi:
        ops += 1
        mid = (lo + hi) // 2
        if sorted_weights[mid] == target:
            return mid, ops
        if sorted_weights[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return None, ops


# weights in pounds: bantamweight=135, featherweight=145, lightweight=155, ...
weights = list(range(115, 266))
idx, ops = weight_class_boundary(weights, 155)  # lightweight cutoff
assert idx == 40, f"expected idx 40, got {idx}"
bound = ceil(log2(len(weights))) + 1
assert ops <= bound, f"ops={ops} exceeded bound {bound}"

# Doubling n adds at most one op
_, ops_small = weight_class_boundary(list(range(1024)), 999)
_, ops_big = weight_class_boundary(list(range(2048)), 1999)
assert ops_big - ops_small <= 1
print(f"log          : weight boundary at idx 40 used {ops} ops (bound = {bound})")

log          : weight boundary at idx 40 used 6 ops (bound = 9)


## Lesson 2.1.1 — O(n) — list comprehension -> iterator

Python `[f for f in roster if f.wins > 10]` is the open-form O(n) scan. The Rust
translation is `roster.iter().filter(|f| f.wins > 10)` — same class, but lazy and
allocation-free until you collect. Counting iterations directly: doubling n
exactly doubles ops.

In [3]:
def unbeaten_count(roster: list[dict]) -> tuple[int, int]:
    """Linear scan: count fighters with zero losses."""
    count, ops = 0, 0
    for f in roster:
        ops += 1
        if f["losses"] == 0:
            count += 1
    return count, ops


roster_a = [
    {"name": f"fighter_{i}", "losses": 0 if i % 4 == 0 else 1} for i in range(1024)
]
roster_b = [
    {"name": f"fighter_{i}", "losses": 0 if i % 4 == 0 else 1} for i in range(2048)
]
count_a, ops_a = unbeaten_count(roster_a)
count_b, ops_b = unbeaten_count(roster_b)
assert ops_a == 1024
assert ops_b == 2048
assert ops_b / ops_a == 2.0
assert count_a == 256  # every 4th
print(f"linear       : ops(roster=1024) = {ops_a}, doubled = {ops_b}, ratio = 2.0")

linear       : ops(roster=1024) = 1024, doubled = 2048, ratio = 2.0


## Lesson 2.3.1 — O(n log n) — `sorted()` -> `sort_unstable`

Python `sorted(roster, key=elo)` is Timsort (stable). Rust `sort_unstable_by_key`
is pdqsort (faster constants). Same Big O class, sub-second constant factor on
large rosters. Here we sanity-check correctness via the order-invariant sum.

In [4]:
import random


def rank_then_total(elos: list[int]) -> int:
    """Sort fighters by Elo then total — order-invariant correctness."""
    return sum(sorted(elos))


random.seed(0)
elos = [random.randint(1800, 2400) for _ in range(1024)]
result = rank_then_total(elos)
expected = sum(elos)
assert result == expected, f"{result} != {expected}"
print(f"linearithmic : rank_then_total returned {result} (order-invariant)")

linearithmic : rank_then_total returned 2142343 (order-invariant)


## O(n²) — nested pair iteration (rivalry pairs)

For every pair of fighters in the roster, check if they've faced each other. The
pair count is exactly `n²` visits. Doubling n quadruples ops.

In [5]:
def rivalry_pair_count(n: int) -> tuple[int, int]:
    """Visit all pairs (i, j) including (i, i)."""
    total, ops = 0, 0
    for _ in range(n):
        for _ in range(n):
            total += 1
            ops += 1
    return total, ops


_, ops_a = rivalry_pair_count(32)
_, ops_b = rivalry_pair_count(64)
assert ops_a == 32 * 32
assert ops_b == 64 * 64
assert ops_b / ops_a == 4.0
print(f"quadratic    : pair_visits(n=32) = {ops_a}, doubled = {ops_b}, ratio = 4.0")

quadratic    : pair_visits(n=32) = 1024, doubled = 4096, ratio = 4.0


## O(2^n) — naive recursion -> memoized

The number of single-elimination tournament brackets for n fighters grows like
Fibonacci. Naive recursion is O(phi^n); `lru_cache` rescues to O(n). The
failure-loop lesson: when your bench is hitting timeouts, memoize before you
blame the algorithm.

In [6]:
from functools import lru_cache


def naive_brackets(n: int) -> int:
    """Naive Fibonacci-like recursion — for n fighters, bracket count."""
    if n < 2:
        return n
    return naive_brackets(n - 1) + naive_brackets(n - 2)


@lru_cache(maxsize=None)
def memo_brackets(n: int) -> int:
    if n < 2:
        return n
    return memo_brackets(n - 1) + memo_brackets(n - 2)


# Correctness against the known sequence
assert [naive_brackets(i) for i in range(8)] == [0, 1, 1, 2, 3, 5, 8, 13]
# Memoized hits O(n) — easy to push n way higher
assert memo_brackets(30) == 832040
print(
    f"exponential  : naive_brackets(20) = {naive_brackets(20)}, memo_brackets(30) = {memo_brackets(30)}"
)

exponential  : naive_brackets(20) = 6765, memo_brackets(30) = 832040


## Iterator fusion — `iterator-fusion-v1`

Python generators are the closest analog to Rust's iterator-fusion. A chained
generator avoids materializing intermediate lists. The fused version computes
the same answer with O(1) extra space.

In [7]:
fighters = list(range(1000))

# Eager (materializes intermediate list)
eager = sum([w * 2 for w in fighters if w % 2 == 0])

# Fused (generator — single pass, no intermediate list)
fused = sum(w * 2 for w in fighters if w % 2 == 0)

assert eager == fused
print(f"fusion       : eager = {eager}, fused = {fused} (equal, fused saves memory)")

fusion       : eager = 499000, fused = 499000 (equal, fused saves memory)


## Transpile preservation — `complexity-preserved-across-transpile-v1`

Conceptual: if `depyler(S)` is semantically equivalent to `S`, the empirical
Big O class is preserved. Constant-factor speedup is *reported*, not asserted.
On iterator-fusion translations (course lesson 2.1.1) a 3-5x speedup is typical.

In [8]:
def transpile_class_preserved(py_class: str, rs_class: str) -> bool:
    return py_class == rs_class


def speedup(py_time: float, rs_time: float) -> float:
    return py_time / rs_time


# depyler keeps the class
assert transpile_class_preserved("O(n)", "O(n)")
assert speedup(40.0, 10.0) == 4.0
print("transpile    : class preserved, expected speedup ~4x on iterator translations")

transpile    : class preserved, expected speedup ~4x on iterator translations


---
**Rust port:** [`m2-empirical/src/lib.rs`](../m2-empirical/src/lib.rs) implements all eight contracts with proptest invariants + criterion benches in `benches/`. Course lessons 2.1.1 (list comp -> iterator), 2.2.1 (dict -> HashMap), 2.3.1 (sorted -> sort_unstable).